<a href="https://colab.research.google.com/github/prasertrak/Advanced-Data-Engineering-and-Applied-Analytics/blob/main/ETL_with_observability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from prometheus_client import Counter, Histogram, Gauge, start_http_server
import logging
import time

# ---------- 1. นิยาม metric ไว้ที่เดียว ใช้ร่วมกันทุก stage ----------
rows_processed = Counter(
    "pipeline_rows_processed_total",
    "จำนวนแถวที่ประมวลผลสำเร็จ",
    ["stage"]                    # label แยกตาม stage
)
rows_failed = Counter(
    "pipeline_rows_failed_total",
    "จำนวนแถวที่ผิดพลาด",
    ["stage", "reason"]          # แยกตามสาเหตุด้วย
)
stage_duration = Histogram(
    "pipeline_stage_duration_seconds",
    "เวลาที่ใช้ในแต่ละ stage",
    ["stage"]
)
queue_size = Gauge(
    "pipeline_queue_size",
    "จำนวนงานที่ค้างอยู่ในคิว ณ ขณะนี้",
    ["stage"]
)

logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)


**Step 1: Ingrstion**

In [ ]:
def ingest_data(source_path: str):
    with stage_duration.labels(stage="ingestion").time():   # จับเวลาอัตโนมัติ
        logging.info(f"เริ่ม ingest จาก {source_path}")
        try:
            raw_data = read_from_source(source_path)
            rows_processed.labels(stage="ingestion").inc(len(raw_data))
            logging.info(f"ingest สำเร็จ {len(raw_data)} แถว")
            return raw_data
        except ConnectionError as e:
            rows_failed.labels(stage="ingestion", reason="connection").inc()
            logging.error(f"ingest ล้มเหลว: {e}")
            raise

**Step 2:Data Cleaning**

def clean_data(raw_data: list):
    with stage_duration.labels(stage="cleaning").time():
        logging.info(f"เริ่ม clean {len(raw_data)} แถว")
        clean_rows = []
        for row in raw_data:
            if is_valid(row):
                clean_rows.append(normalize(row))
                rows_processed.labels(stage="cleaning").inc()
            else:
                rows_failed.labels(stage="cleaning", reason="invalid_schema").inc()
                logging.warning(f"พบแถวผิด schema: {row.get('id', 'unknown')}")

        queue_size.labels(stage="cleaning").set(len(clean_rows))
        logging.info(f"clean เสร็จ ผ่าน {len(clean_rows)}/{len(raw_data)} แถว")
        return clean_rows

**Step 3: Transform**

def transform_data(clean_rows: list):
    with stage_duration.labels(stage="transform").time():
        logging.info(f"เริ่ม transform {len(clean_rows)} แถว")
        try:
            transformed = apply_business_logic(clean_rows)
            rows_processed.labels(stage="transform").inc(len(transformed))
            logging.info("transform สำเร็จ")
            return transformed
        except Exception as e:
            rows_failed.labels(stage="transform", reason="logic_error").inc()
            logging.error(f"transform ล้มเหลว: {e}")
            raise

**รันทั้ง pipeline + เปิด endpoint ให้ Prometheus มาดึง**

if __name__ == "__main__":
    start_http_server(8000)   # เปิด http://host:8000/metrics ทิ้งไว้

    raw = ingest_data("s3://bucket/sales.csv")
    clean = clean_data(raw)
    result = transform_data(clean)  

สิ่งสำคัญ: โค้ดข้างบนไม่ได้ส่งอะไรไปหา Prometheus โดยตรงเลย — prometheus_client ทำแค่ 2 อย่าง:

เก็บตัวเลขไว้ในหน่วยความจำของโปรเซส (in-memory) ทุกครั้งที่เรียก .inc(), .set(), .time()
เปิด HTTP endpoint (start_http_server(8000)) ที่คอยตอบค่าตัวเลขล่าสุดเมื่อมีใครมาถาม

ส่วน Prometheus server ต่างหากที่เป็นฝ่ายเดินเข้ามาดึง (ตามที่คุยกันก่อนหน้า) — ทุก 15 วินาที มันจะยิง GET ไปที่ http://your-host:8000/metrics แล้วจะเห็นข้อความดิบแบบนี้:

pipeline_rows_processed_total{stage="ingestion"} 15420
pipeline_rows_processed_total{stage="cleaning"} 15100
pipeline_rows_processed_total{stage="transform"} 15100
pipeline_rows_failed_total{stage="cleaning",reason="invalid_schema"} 320
pipeline_stage_duration_seconds_sum{stage="cleaning"} 4.82